## In this Notebook we are going to analyse and prepare our data for the modeling phase.


In [63]:
import pandas as pd
from datetime import datetime
import re


In [64]:
import numpy as np
import scipy

import math
import random
import sklearn
from nltk.corpus import stopwords
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

### Loading our data

In [65]:
interactions_df = pd.read_csv('data/cleaned_users_interactions.csv', index_col=0)
interactions_df.head(10)

,timestamp,eventType,contentId,personId,sessionId,userAgent,userRegion,userCountry,datetime,month,countryName,userCountryRegion
0,1465413032,VIEW,-3499919498720038879,-8845298781299428018,1264196770339959068,NaN,NaN,NaN,2016-06-08 21:10:32,2016-06,NaN,NaN
1,1465412560,VIEW,8890720798209849691,-1032019229384696495,3621737643587579081,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,NY,US,2016-06-08 21:02:40,2016-06,United States,US-NY
2,1465416190,VIEW,310515487419366995,-1130272294246983140,2631864456530402479,NaN,NaN,NaN,2016-06-08 22:03:10,2016-06,NaN,NaN
3,1465413895,FOLLOW,310515487419366995,344280948527967603,-3167637573980064150,NaN,NaN,NaN,2016-06-08 21:24:55,2016-06,NaN,NaN
4,1465412290,VIEW,-7820640624231356730,-445337111692715325,5611481178424124714,NaN,NaN,NaN,2016-06-08 20:58:10,2016-06,NaN,NaN
5,1465413742,VIEW,310515487419366995,-8763398617720485024,1395789369402380392,Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebK...,MG,BR,2016-06-08 21:22:22,2016-06,Brazil,BR-MG
6,1465415950,VIEW,-8864073373672512525,3609194402293569455,1143207167886864524,NaN,NaN,NaN,2016-06-08 21:59:10,2016-06,NaN,NaN
7,1465415066,VIEW,-1492913151930215984,4254153380739593270,8743229464706506141,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,SP,BR,2016-06-08 21:44:26,2016-06,Brazil,BR-SP
8,1465413762,VIEW,310515487419366995,344280948527967603,-3167637573980064150,NaN,NaN,NaN,2016-06-08 21:22:42,2016-06,NaN,NaN
9,1465413771,VIEW,3064370296170038610,3609194402293569455,1143207167886864524,NaN,NaN,NaN,2016-06-08 21:22:51,2016-06,NaN,NaN


In [66]:
articles_df = pd.read_csv('data/shared_articles.csv')
articles_df.head(5)

,timestamp,eventType,contentId,authorPersonId,authorSessionId,authorUserAgent,authorRegion,authorCountry,contentType,url,title,text,lang
0,1459192779,CONTENT REMOVED,-6451309518266745024,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.nytimes.com/2016/03/28/business/dea...,"Ethereum, a Virtual Currency, Enables Transact...",All of this work is still very early. The firs...,en
1,1459193988,CONTENT SHARED,-4110354420726924665,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.nytimes.com/2016/03/28/business/dea...,"Ethereum, a Virtual Currency, Enables Transact...",All of this work is still very early. The firs...,en
2,1459194146,CONTENT SHARED,-7292285110016212249,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://cointelegraph.com/news/bitcoin-future-w...,Bitcoin Future: When GBPcoin of Branson Wins O...,The alarm clock wakes me at 8:00 with stream o...,en
3,1459194474,CONTENT SHARED,-6151852268067518688,3891637997717104548,-1457532940883382585,NaN,NaN,NaN,HTML,https://cloudplatform.googleblog.com/2016/03/G...,Google Data Center 360° Tour,We're excited to share the Google Data Center ...,en
4,1459194497,CONTENT SHARED,2448026894306402386,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,https://bitcoinmagazine.com/articles/ibm-wants...,"IBM Wants to ""Evolve the Internet"" With Blockc...",The Aite Group projects the blockchain market ...,en


### Data munging

#### 
As there are different interactions types, we associate them with a weight or strength, assuming that, for example, a comment in an article indicates a higher interest of the user on the item than a like, or than a simple view.

In [67]:
event_type_strength = {
   'VIEW': 1.0,
   'LIKE': 2.0, 
   'BOOKMARK': 2.5, 
   'FOLLOW': 3.0,
   'COMMENT CREATED': 4.0,  
}

interactions_df['eventStrength'] = interactions_df['eventType'].apply(lambda x: event_type_strength[x])

In [68]:
users_interactions_count_df = interactions_df.groupby(['personId', 'contentId']).size().groupby('personId').size()
print('# users: %d' % len(users_interactions_count_df))
users_with_enough_interactions_df = users_interactions_count_df[users_interactions_count_df >= 5].reset_index()[['personId']]
print('# users with at least 5 interactions: %d' % len(users_with_enough_interactions_df))

# users: 1894
# users with at least 5 interactions: 1138


In [69]:
print('# of interactions: %d' % len(interactions_df))
interactions_from_selected_users_df = interactions_df.merge(users_with_enough_interactions_df, 
               how = 'right',
               left_on = 'personId',
               right_on = 'personId')
print('# of interactions from users with at least 5 interactions: %d' % len(interactions_from_selected_users_df))

# of interactions: 71904
# of interactions from users with at least 5 interactions: 69466


In [70]:
import pandas as pd
import math

# Define the smoothing function
def smooth_user_preference(x):
    return math.log(1 + x, 2)

# Extract the last interaction's datetime for each (personId, contentId)
last_interactions_df = interactions_from_selected_users_df \
    .groupby(['personId', 'contentId'])['datetime'].max().reset_index()

# Aggregate eventStrength and apply smoothing
interactions_full_df = interactions_from_selected_users_df \
    .groupby(['personId', 'contentId'])['eventStrength'].sum() \
    .apply(smooth_user_preference).reset_index()

# Merge the last datetime information into interactions_full_df
interactions_full_df = interactions_full_df.merge(
    last_interactions_df,
    on=['personId', 'contentId'],
    how='left'
)

# Print the number of unique interactions
print('# of unique user/item interactions: %d' % len(interactions_full_df))

# Display the first 10 rows of the resulting dataframe
print(interactions_full_df.head(10))


# of unique user/item interactions: 38885
              personId            contentId  eventStrength  \
0 -9223121837663643404 -8949113594875411859       1.000000   
1 -9223121837663643404 -8377626164558006982       1.000000   
2 -9223121837663643404 -8208801367848627943       1.000000   
3 -9223121837663643404 -8187220755213888616       1.000000   
4 -9223121837663643404 -7423191370472335463       3.169925   
5 -9223121837663643404 -7331393944609614247       1.000000   
6 -9223121837663643404 -6872546942144599345       1.000000   
7 -9223121837663643404 -6728844082024523434       1.000000   
8 -9223121837663643404 -6590819806697898649       1.000000   
9 -9223121837663643404 -6558712014192834002       1.584963   

              datetime  
0  2016-05-05 14:42:07  
1  2016-09-15 13:25:07  
2  2016-07-28 13:51:42  
3  2016-07-06 18:51:37  
4  2016-11-17 10:56:18  
5  2016-05-17 14:00:17  
6  2017-02-21 12:50:00  
7  2017-01-24 11:06:27  
8  2017-01-24 11:06:25  
9  2016-06-03 17:42:44  


In [71]:
# Ensure the datetime column is in datetime format
interactions_full_df['datetime'] = pd.to_datetime(interactions_full_df['datetime'])

# Sort the dataset by datetime in ascending order
interactions_full_df = interactions_full_df.sort_values(by='datetime')

# Calculate the index for 80% of the data
split_index = int(len(interactions_full_df) * 0.8)

# Split the data into train (first 80%) and test (last 20%)
interactions_train_df = interactions_full_df.iloc[:split_index]
interactions_test_df = interactions_full_df.iloc[split_index:]

interactions_test_df = interactions_test_df[np.isin(interactions_test_df.personId, interactions_train_df.personId)]
interactions_train_df = interactions_train_df[np.isin(interactions_train_df.personId, interactions_test_df.personId)]
train_len = len(interactions_train_df)
test_len = len(interactions_test_df)
full_len = train_len + test_len

# Print the sizes of the train and test sets
print(f"Training set size: {len(interactions_train_df)}")
print(f"Testing set size: {len(interactions_test_df)}")

# Optionally, save the splits to separate files
interactions_train_df.to_csv('data/interactions_train.csv', index=False)
interactions_test_df.to_csv('data/interactions_test.csv', index=False)

Training set size: 28833
Testing set size: 7548


## Evaluation

In [72]:
interactions_full_indexed_df = interactions_full_df.set_index('personId')
interactions_train_indexed_df = interactions_train_df.set_index('personId')
interactions_test_indexed_df = interactions_test_df.set_index('personId')

In [73]:
def get_items_interacted(person_id, interactions_df):
    # Get the user's data and merge in the movie information.
    interacted_items = interactions_df.loc[person_id]['contentId']
    return set(interacted_items if type(interacted_items) == pd.Series else [interacted_items])

In [74]:
# Top-N accuracy metrics consts
EVAL_RANDOM_SAMPLE_NON_INTERACTED_ITEMS = 100

class ModelEvaluator:

    def get_not_interacted_items_sample(self, person_id, sample_size, seed=42):
        interacted_items = get_items_interacted(person_id, interactions_full_indexed_df)
        all_items = set(articles_df['contentId'])
        non_interacted_items = all_items - interacted_items

        random.seed(seed)
        non_interacted_items_sample = random.sample(non_interacted_items, sample_size)
        return set(non_interacted_items_sample)

    def _verify_hit_top_n(self, item_id, recommended_items, topn):        
        try:
            index = next(i for i, c in enumerate(recommended_items) if c == item_id)
        except:
            index = -1
        hit = int(index in range(0, topn))
        return hit, index

    def evaluate_model_for_user(self, model, person_id):
        # Getting the items in test set
        interacted_values_testset = interactions_test_indexed_df.loc[person_id]
        if type(interacted_values_testset['contentId']) == pd.Series:
            person_interacted_items_testset = set(interacted_values_testset['contentId'])
        else:
            person_interacted_items_testset = set([int(interacted_values_testset['contentId'])])  
        interacted_items_count_testset = len(person_interacted_items_testset) 

        # Getting a ranked recommendation list from a model for a given user
        person_recs_df = model.recommend_items(person_id, 
                                               items_to_ignore=get_items_interacted(person_id, 
                                                                                    interactions_train_indexed_df), 
                                               topn=10000000000)

        hits_at_5_count = 0
        hits_at_10_count = 0
        # For each item the user has interacted in test set
        for item_id in person_interacted_items_testset:
            # Getting a random sample (100) items the user has not interacted 
            # (to represent items that are assumed to be no relevant to the user)
            non_interacted_items_sample = self.get_not_interacted_items_sample(person_id, 
                                                                          sample_size=EVAL_RANDOM_SAMPLE_NON_INTERACTED_ITEMS, 
                                                                          seed=item_id%(2**32))

            # Combining the current interacted item with the 100 random items
            items_to_filter_recs = non_interacted_items_sample.union(set([item_id]))

            # Filtering only recommendations that are either the interacted item or from a random sample of 100 non-interacted items
            valid_recs_df = person_recs_df[person_recs_df['contentId'].isin(items_to_filter_recs)]                    
            valid_recs = valid_recs_df['contentId'].values
            # Verifying if the current interacted item is among the Top-N recommended items
            hit_at_5, index_at_5 = self._verify_hit_top_n(item_id, valid_recs, 5)
            hits_at_5_count += hit_at_5
            hit_at_10, index_at_10 = self._verify_hit_top_n(item_id, valid_recs, 10)
            hits_at_10_count += hit_at_10

        # Recall is the rate of the interacted items that are ranked among the Top-N recommended items, 
        # when mixed with a set of non-relevant items
        recall_at_5 = hits_at_5_count / float(interacted_items_count_testset)
        recall_at_10 = hits_at_10_count / float(interacted_items_count_testset)        
        person_metrics = {'hits@5_count':hits_at_5_count, 
                          'hits@10_count':hits_at_10_count, 
                          'interacted_count': interacted_items_count_testset,
                          'recall@5': recall_at_5,
                          'recall@10': recall_at_10}
        return person_metrics

    def evaluate_model(self, model):
        # print('Running evaluation for users')
        people_metrics = []
        for idx, person_id in enumerate(list(interactions_test_indexed_df.index.unique().values)):
            # if idx % 100 == 0 and idx > 0:
            #    print('%d users processed' % idx)
            person_metrics = self.evaluate_model_for_user(model, person_id)  
            person_metrics['_person_id'] = person_id
            people_metrics.append(person_metrics)
        print('%d users processed' % idx)

        detailed_results_df = pd.DataFrame(people_metrics) \
                            .sort_values('interacted_count', ascending=False)
        
        global_recall_at_5 = detailed_results_df['hits@5_count'].sum() / float(detailed_results_df['interacted_count'].sum())
        global_recall_at_10 = detailed_results_df['hits@10_count'].sum() / float(detailed_results_df['interacted_count'].sum())
        
        global_metrics = {'modelName': model.get_model_name(),
                          'recall@5': global_recall_at_5,
                          'recall@10': global_recall_at_10}    
        return global_metrics, detailed_results_df

# Instantiate the ModelEvaluator
model_evaluator = ModelEvaluator()


In [75]:
# Compute the most popular items
item_popularity_df = interactions_full_df.groupby('contentId')['eventStrength'].sum().sort_values(ascending=False).reset_index()

class PopularityRecommender:
    
    MODEL_NAME = 'Popularity'
    
    def __init__(self, popularity_df, items_df=None):
        self.popularity_df = popularity_df
        self.items_df = items_df
        
    def get_model_name(self):
        return self.MODEL_NAME
        
    def recommend_items(self, user_id, items_to_ignore=[], topn=10, verbose=False):
        # Recommend the most popular items that the user hasn't seen yet.
        recommendations_df = self.popularity_df[~self.popularity_df['contentId'].isin(items_to_ignore)] \
                               .sort_values('eventStrength', ascending=False) \
                               .head(topn)

        if verbose:
            if self.items_df is None:
                raise Exception('"items_df" is required in verbose mode')

            recommendations_df = recommendations_df.merge(self.items_df, how='left', on='contentId')[['eventStrength', 'contentId', 'title', 'url']]
        
        return recommendations_df


In [76]:
# Instantiate the PopularityRecommender
popularity_model = PopularityRecommender(item_popularity_df, articles_df)

# Test the recommender for a specific user
user_id = interactions_test_df['personId'].iloc[0]  # Example user ID from test data
recommended_items = popularity_model.recommend_items(user_id, items_to_ignore=get_items_interacted(user_id, interactions_train_indexed_df), topn=5, verbose=True)

# Display recommendations
print(recommended_items)


   eventStrength            contentId  \
0     307.733799 -4029704725707465084   
1     233.762157 -6783772548752091658   
2     228.024567  -133139342397538859   
3     197.107608 -8208801367848627943   
4     189.044680  8224860111193157980   

                                               title  \
0  Former Google career coach shares a visual tri...   
1                   Livro: Retrospectivas Divertidas   
2  Novo workaholic trabalha, pratica esportes e t...   
3  Ray Kurzweil: The world isn't getting worse - ...   
4  Psicóloga de Harvard diz que as pessoas julgam...   

                                                 url  
0  http://www.cnbc.com/2016/12/21/former-google-c...  
1  http://www.caroli.org/livro-retrospectivas-div...  
2  http://gq.globo.com/Prazeres/Poder/Carreira/no...  
3  http://www.geekwire.com/2016/ray-kurzweil-worl...  
4  http://www.jornaldoempreendedor.com.br/destaqu...  


In [77]:
popularity_global_metrics, pop_detailed_results_df = model_evaluator.evaluate_model(popularity_model)

# Now you can print global metrics and view the detailed results
print('\nGlobal metrics:\n%s' % popularity_global_metrics)
pop_detailed_results_df.head(10)


C:\Users\21623\AppData\Local\Temp\ipykernel_8568\3158618790.py:12: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  non_interacted_items_sample = random.sample(non_interacted_items, sample_size)


917 users processed

Global metrics:
{'modelName': 'Popularity', 'recall@5': 0.28312135665076843, 'recall@10': 0.424483306836248}


,hits@5_count,hits@10_count,interacted_count,recall@5,recall@10,_person_id
26,24,48,220,0.109091,0.218182,3609194402293569455
77,32,45,108,0.296296,0.416667,-2979881261169775358
12,18,29,88,0.204545,0.329545,1116121227607581999
68,24,36,80,0.300000,0.450000,8698209668239173600
81,15,23,80,0.187500,0.287500,692689608292948411
8,9,15,72,0.125000,0.208333,-3596626804281480007
115,21,32,69,0.304348,0.463768,-9016528795238256703
124,16,29,64,0.250000,0.453125,2760160224512620259
100,26,36,64,0.406250,0.562500,4313045637915476309
22,16,24,59,0.271186,0.406780,-3203894957285229214
